# 主に東北大学GLCなどに開示されている留学情報をスクレイピングするアルゴリズムを作った：

In [1]:
import pandas as pd
import numpy as np
import datetime
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
#import lightgbm as lgb
import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.request import Request, urlopen
#import optuna.integration.lightgbm as lgb_o
from itertools import combinations, permutations
import matplotlib.pyplot as plt


In [2]:
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

random.choice(USER_AGENTS)

'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15'

In [3]:
url = "https://www.insc.tohoku.ac.jp/japanese/studyabroad/exploring/program/"
headers = {'User-Agent': random.choice(USER_AGENTS)}
html = requests.get(url, headers=headers)
html.encoding = html.apparent_encoding 
# html.encoding = "EUC-JP"

In [29]:
soup = BeautifulSoup(html.text, "html.parser")

In [5]:
print(soup.prettify())

<!DOCTYPE html>
<html lang="ja">
 <head>
  <meta charset="utf-8"/>
  <title>
   海外体験プログラム 募集情報 | 東北大学
  </title>
  <meta content="短期海外研修プログラム(SAP)のほか、協定校等主催のショートプログラムや大学間協定に基づく交換留学、大学院生のための留学を紹介しています。" name="description"/>
  <meta content="Tohoku, University, International, Student, Exchange, Foreign, Global,Undergraduate, Graduate,English,東北、大学、大学院、留学生、留学、英語、語学、インターナショナル、グローバル、海外" name="keywords"/>
  <meta content="all" name="robots"/>
  <meta content="ja_JP" property="og:locale">
   <meta content="東北大学グローバルラーニングセンター-Tohoku University" property="og:site_name"/>
   <meta content="  海外体験プログラム 募集情報 | 東北大学" property="og:title"/>
   <meta content="短期海外研修プログラム(SAP)のほか、協定校等主催のショートプログラムや大学間協定に基づく交換留学、大学院生のための留学を紹介しています。" property="og:description"/>
   <meta content="website" property="og:type">
    <link href="/common/img/favicon.ico" rel="shortcut icon"/>
    <meta content="width=device-width,initial-scale=1" name="viewport"/>
    <link href="/common/css/custom_reset.css" media="all" rel="st

In [8]:
soup.find('h1')

<h1 class="title"><span>  海外体験プログラム 募集情報</span></h1>

In [30]:
# テーブルの取得
table = soup.find('table', class_='programTable')
rows = table.find_all('tr')
rows

[<tr>
 <th>プログラム名</th>
 <td colspan="3">
 <a href="https://nkries.jp/" target="_blank">国際学生交流プログラム Nakatani RIES 2026</a></td>
 </tr>,
 <tr>
 <th>派遣先</th>
 <td colspan="3">
 <p id="Nakatani26">【米国】ジョージア工科大学</p></td>
 </tr>,
 <tr>
 <th>日程</th>
 <td colspan="3">2026.8.9~2026.9.27</td>
 </tr>,
 <tr>
 <th>概要</th>
 <td colspan="3">【概要】公益財団法人 中谷医工計測技術振興財団が費用の多くを助成する夏季短期留学プログラムです。工学分野で世界でも有名なジョージア工科大学での研究活動を通じ、研究者を目指す学生や大学教授とネットワークを築きながら、今後のキャリアを考え拡げることができる充実したプログラムです。<br/>
 <br/>
 【プログラム内容】<br/>
 ・留学期間：2026年8月9日（日）～9月27日（日）<br/>
 ・渡航前オリエンテーション（国内）<br/>
 ・ジョージア工科大学におけるオリエンテーション＆研究室におけるリサーチインターンシップ：上記の渡航日程で各学生の専攻分野、興味のある分野に基づき研究室に配属されます。研究が初めての参加者もメンター（大学院生、ポスドク）がアサインされて日々の研究活動をサポートしてくれるようになっています。<br/>
 ・ジョージア工科大学の学生との共同のポスターセッション<br/>
 ・帰国発表会（国内）<br/>
 ・※上記を含め、当財団が指定する全日程に参加することが条件です<br/>
 <br/>
 【助成内容】<br/>
 ・研修費用（日本でのオリエンテーション、米国ジョージア工科大学での研究活動）<br/>
 ・食費の一部（Stipendsの支給）<br/>
 ・渡航費ならびに国内外での移動費<br/>
 ・滞在費（日本から渡米時の前泊、米国滞在中）<br/>
 ・海外旅行保険代（＊別途本学の規定により「付帯海学」へ加入する必要があり、こちらは自己負担となります）、Visa 取得費用<b

In [27]:
data = []
for row in rows:
    header = row.find('th').get_text(strip=True) # ヘッダー（項目名）
    cell = row.find('td')
    data.append({'Header':header})
    

In [28]:
data

[{'Header': 'プログラム名'},
 {'Header': '派遣先'},
 {'Header': '日程'},
 {'Header': '概要'},
 {'Header': '応募〆切'},
 {'Header': '必要書類'},
 {'Header': '備考'}]

In [31]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 1. スクレイピング対象のURL
url = "https://www.insc.tohoku.ac.jp/japanese/studyabroad/exploring/program/"

# 2. アクセス設定（User-Agentを設定しないと拒否されることがあるため設定します）
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

def get_program_data(url):
    print(f"Fetching: {url}")
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status() # エラーなら例外を発生
    except Exception as e:
        print(f"Error fetching page: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # ページ内のすべての class="programTable" を探す
    tables = soup.find_all('table', class_='programTable')
    print(f"Found {len(tables)} programs.")
    
    all_programs = []

    for table in tables:
        rows = table.find_all('tr')
        program_info = {}
        
        for row in rows:
            # ヘッダー（項目名）がない行はスキップ
            th = row.find('th')
            if not th:
                continue
                
            key = th.get_text(strip=True) # 列名（プログラム名、派遣先など）
            td = row.find('td')
            
            # テキストの取得（改行を維持）
            text_content = td.get_text(separator='\n', strip=True)
            
            # リンクがある場合はURLもテキストに追加しておく
            links = td.find_all('a', href=True)
            if links:
                link_texts = []
                for a in links:
                    # 相対パスの場合は絶対パスに変換（必要であれば）
                    href = a['href']
                    if href.startswith('/'):
                        href = "https://www.insc.tohoku.ac.jp" + href
                    link_texts.append(f"[{a.get_text(strip=True)}]({href})")
                
                # テキストの後ろにリンク情報を追記
                text_content += "\n\n<関連リンク>\n" + "\n".join(link_texts)

            program_info[key] = text_content
        
        all_programs.append(program_info)
        
    return all_programs

# --- 実行 ---
data_list = get_program_data(url)

# DataFrameに変換
if data_list:
    df = pd.DataFrame(data_list)
    
    # 列の順序を整える（任意の順序に変更可能）
    # データに含まれるカラムだけを指定して並べ替え
    columns_order = ['プログラム名', '派遣先', '日程', '概要', '応募〆切', '必要書類', '備考']
    existing_cols = [c for c in columns_order if c in df.columns]
    # 未定義のカラムがあれば後ろに追加
    remaining_cols = [c for c in df.columns if c not in columns_order]
    df = df[existing_cols + remaining_cols]

    print("--- 取得結果 (最初の3行) ---")
    print(df.head(3))
    
    # Excelに出力
    output_file = "tohoku_programs.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\nSaved to {output_file}")
else:
    print("データが見つかりませんでした。ページ構造が異なっているか、JavaScriptでロードされている可能性があります。")

Fetching: https://www.insc.tohoku.ac.jp/japanese/studyabroad/exploring/program/
Found 137 programs.
--- 取得結果 (最初の3行) ---
                                              プログラム名  \
0  国際学生交流プログラム Nakatani RIES 2026\n\n<関連リンク>\n[国際...   
1  Online German Language Program\n\n<関連リンク>\n[On...   
2  2026 NTU Plus Academy Research & Culture Progr...   

                             派遣先                                日程  \
0                  【米国】ジョージア工科大学                2026.8.9~2026.9.27   
1  【ドイツ】ヨハネスグーテンベルク大学マインツ（オンライン）              2026.2.23 ~ 2026.3.6   
2                     【台湾】国立台湾大学  ※応募終了しました\n2026.2.24 ~ 2026.3.28   

                                                  概要  \
0  【概要】公益財団法人 中谷医工計測技術振興財団が費用の多くを助成する夏季短期留学プログラムで...   
1  【概要】ヨハネスグーテンベルク大学マインツが主催する、オンライン型のドイツ語研修プログラムで...   
2  国立台湾大学（NTU）が主催する5週間の研究型プログラムです。NTUの研究室にてNTU教員の...   

                                                応募〆切  \
0  【学内応募締切】2月9日（月）正午\n参加希望の方は上記の期限までに以下必要書類の項目にある...   
1  【学内締切】\n2026年1月6日（火）正午\n・参加希望の方は上

In [32]:
df

,プログラム名,派遣先,日程,概要,応募〆切,必要書類,備考
0,国際学生交流プログラム Nakatani RIES 2026\n\n<関連リンク>\n[国際...,【米国】ジョージア工科大学,2026.8.9~2026.9.27,【概要】公益財団法人 中谷医工計測技術振興財団が費用の多くを助成する夏季短期留学プログラムで...,【学内応募締切】2月9日（月）正午\n参加希望の方は上記の期限までに以下必要書類の項目にある...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,【科目履修について】\n・全学教育科目（「海外短期研修（発展B）」の履修が可能です。\n※履...
1,Online German Language Program\n\n<関連リンク>\n[On...,【ドイツ】ヨハネスグーテンベルク大学マインツ（オンライン）,2026.2.23 ~ 2026.3.6,【概要】ヨハネスグーテンベルク大学マインツが主催する、オンライン型のドイツ語研修プログラムで...,【学内締切】\n2026年1月6日（火）正午\n・参加希望の方は上記の期限までに以下必要書類...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,・全学教育科目（「海外短期研修（発展A）」）の履修が可能です。\n・履修登録は留学生課で行い...
2,2026 NTU Plus Academy Research & Culture Progr...,【台湾】国立台湾大学,※応募終了しました\n2026.2.24 ~ 2026.3.28,国立台湾大学（NTU）が主催する5週間の研究型プログラムです。NTUの研究室にてNTU教員の...,【学内締切】\n12月1日（月）正午\n・上記の期日までに以下に記載の必要書類を用意したうえ...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,【科目履修について】\n全学教育科目（「海外短期研修（発展Ｂ）」の履修が可能です。\nただし...
3,SMART LYON\n\n<関連リンク>\n[SMART LYON](https://ww...,【フランス】国立応用科学院リヨン校,※応募終了しました\n２026.3.9 ~ 2026.3.20,【概要】フランスの国立応用科学院リヨン校 (INSA-Lyon) 主催の、理工学を学ぶ学生対...,【学内締切】\n11月20日（木）正午\n→\n応募期間を12月4日（木）正午に延長しました...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,全学教育科目（「海外短期研修（発展A）」）の履修が可能です\n※科目履修をする、しないに関わ...
4,Virtual CommTECH Exploration 2026\n\n<関連リンク>\n...,【インドネシア】セプル・ノーペンバー工科大学（オンライン）,※応募終了しました\n2026.1.5 ~ 2026.1.16,【概要】インドネシアのセプル・ノーペンバー工科大学(ITS)が主催する、ITSとインドネシア...,【学内応募締切】12月8日（月）正午\n参加希望の方は上記の期限までに以下必要書類の項目にあ...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,【科目履修について】\n※全学教育科目（「海外短期研修（発展Ａ）」の履修が可能です。\n※履...
...,...,...,...,...,...,...,...
132,HKUST International Summer Exchange Program (I...,【香港】香港科技大学,※応募終了※2024.6.17 ~ 2024.8.10,【概要】香港科技大学(HKUST)が授業料免除で協定校の学生のために実施する国際夏季交換プロ...,【学内応募締切】2月19日（月）正午\n参加希望の方は上記の期限までに以下必要書類の項目にあ...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,【科目履修について】\n・学部生の場合はこの期間に他の科目を履修していない場合、全学教育科目...
133,International Summer Program (ISP) 2024\n\n<関連...,【ドイツ】ドルトムント工科大学,※応募終了※2024.6.3 ~ 2024.7.31,【概要】ドイツのドルトムント工科大学が協定校の学生のみを招待し開催する約2か月のサマープログ...,【学内締切】\n2月12日（月）までに以下の提出書類を現地研修型海外体験プログラム・ショート...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,【科目履修について】\n・学部生の場合はこの期間に他の科目を履修していない場合、全学教育科目...
134,UBC Vancouver Summer Program 2024\n\n<関連リンク>\n...,【カナダ】ブリティッシュ・コロンビア大学,※応募終了※①2024.6.7 ~ 2024.7.7 ②2024.7.12 ~ 2024....,【概要】カナダのバンクーバーに位置するブリティッシュ・コロンビア大学が主催する4週間のサマー...,【学内締切】\n①3月18日（月）正午までに以下の提出書類を現地研修型海外体験プログラム・シ...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,【科目履修について】\n・学部生の場合はこの期間に他の科目を履修していない場合、全学教育科目...
135,国際学生交流プログラム Nakatani RIES 2024\n\n<関連リンク>\n[国際...,【米国】ジョージア工科大学,※応募終了※2024.8.4 ~ 2024.9.22,【概要】公益財団法人 中谷医工計測技術振興財団が費用の多くを助成する夏季短期留学プログラムで...,【学内応募締切】2月15日（木）正午\n参加希望の方は上記の期限までに以下必要書類の項目にあ...,・参加希望理由書（海外体験プログラム用所定様式）\n参加希望理由書\n\n<関連リンク>\n...,・全学教育科目（「海外短期研修（発展B）」の履修が可能\n＊履修登録は留学生課で行います。


In [ ]:
# kaizen 

